In [ ]:
import tensorflow as tf

from tensorflow_model_optimization.python.core.keras.compat import keras

In [ ]:
# 1. Define the Keras Model Architecture
# PyTorch uses (channels, height, width), but Keras defaults to (height, width, channels)
model = keras.Sequential([
    # Input Layer (32x32 images with 3 channels)
    keras.layers.Input(shape=(32, 32, 3)),
    
    # conv1
    keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same'),
    # conv2
    keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
    # conv3
    keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
    # conv4
    keras.layers.Conv2D(filters=48, kernel_size=(3, 3), activation='relu', padding='same'),
    # conv5
    keras.layers.Conv2D(filters=48, kernel_size=(3, 3), activation='relu', padding='same'),
    # conv6
    keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same'),
    
    # conv7: FusedMaxPoolConv2dReLU (Pooling is applied BEFORE the convolution in ai8x)
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(filters=16, kernel_size=(3, 3), activation='relu', padding='same'),
    
    # conv8: FusedMaxPoolConv2dReLU
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(filters=8, kernel_size=(3, 3), activation='relu', padding='same'),
    
    # Flatten the 3D tensor to 1D before passing to Dense layer
    keras.layers.Flatten(),
    
    # Fully Connected Layer (fc) - 10 output classes for CIFAR-10
    keras.layers.Dense(10) 
])

# Display the model summary to verify the shape transitions
model.summary()

In [ ]:
# 1. Load Data
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# 2. Split Training Data (50,000 total) into Train (45,000) and Val (5,000)
# We take the first 45k for training, last 5k for validation

# Normalize pixel values
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Define Transformations
train_transform = keras.Sequential([
    keras.layers.ZeroPadding2D(padding=4),
    keras.layers.RandomCrop(height=32, width=32),
    keras.layers.RandomFlip("horizontal"),
])

BATCH_SIZE = 64

# 3. Create Datasets
train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=50000)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (train_transform(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)




# def make_step_decay_schedule(decay_epochs, factor):
#     decay_epochs = set(decay_epochs)

#     def schedule(epoch, lr):
#         if epoch in decay_epochs:
#             return lr * factor
#         return lr

#     return schedule

# lr_callback = keras.callbacks.LearningRateScheduler(
#     make_step_decay_schedule(decay_epochs=[100, 150, 90], factor=0.1),
#     verbose=1
# )




opt = keras.optimizers.Adam(learning_rate=3e-4)
# 4. Compile and Train
model.compile(
    optimizer=opt,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.fit(train_dataset, epochs=10, validation_split=0.05)

model.save("./cifartest.keras")
# 5. Final Evaluation (Independent Test Set)
test_loss, test_acc = model.evaluate(test_dataset, verbose=2)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")

In [ ]:
import tensorflow_model_optimization as tfmot

quantize_model = tfmot.quantization.keras.quantize_model

# q_aware stands for for quantization aware.
q_aware_model = quantize_model(model)

# `quantize_model` requires a recompile.
q_aware_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

q_aware_model.summary()